In [ ]:
import json

import polars as pl

import wandb

In [ ]:
DATASET = "icon"
api = wandb.Api()

# Get runs with a specific tag
runs = api.runs(
    "feik/genpp",  # Replace with your entity/project
    filters={"tags": {"$all": ["final", DATASET], "$nin": ["deprecated", "extra"]}},
)

wandb: [wandb.Api()] Loaded credentials for https://api.wandb.ai from /home/feik/.netrc.


In [3]:
for run in runs:
    print(run.url, run.name, run.tags)

https://wandb.ai/feik/genpp/runs/iy9kmiv2 vocal-valley-1061 ['direct', 'extmfeik', 'final', 'fm_uvit_direct', 'icon', 'icon_full_pad_xy']
https://wandb.ai/feik/genpp/runs/fm8sfy6b restful-firebrand-1065 ['emos', 'extmfeik', 'final', 'icon', 'icon_cut']
https://wandb.ai/feik/genpp/runs/ql4z0tt0 super-music-1068 ['direct', 'extmfeik', 'final', 'fm_unet_direct', 'icon', 'icon_full_pad_xy']
https://wandb.ai/feik/genpp/runs/24yqcvzc peach-lake-1070 ['extmfeik', 'final', 'fm_uvit_noise', 'icon', 'icon_full_pad_xy', 'indirect']
https://wandb.ai/feik/genpp/runs/n5klic9q polished-snowball-1071 ['extmfeik', 'final', 'fm_unet_noise', 'icon', 'icon_full_pad_xy', 'indirect']
https://wandb.ai/feik/genpp/runs/qmge5ywq dry-universe-1072 ['drn', 'extmfeik', 'final', 'icon', 'icon_full_minmax']
https://wandb.ai/feik/genpp/runs/9c6zdg7p astral-breeze-1073 ['cnn_chen_noise', 'extmfeik', 'final', 'icon', 'icon_full_pad_x', 'indirect', 'multiscale_energy_score']
https://wandb.ai/feik/genpp/runs/q6sdblyf leg

In [9]:
"""
Parse WandB runs into a Polars DataFrame.
Handles nested metrics with optional method keys (ECC, GCA, etc.)
and includes dataset level (val/test).
"""
rows = []

for run in runs:
    run_id = run.id
    summary = run.summary._json_dict
    datasets = [name for name in ["val", "test"] if isinstance(summary.get(name), dict)]

    if not datasets:
        print(f"Run {run_id} has no 'val' or 'test' metrics, skipping.")
        continue

    json_config = json.loads(run.json_config)
    model_name = json_config["name"]["value"]

    for dataset in datasets:
        metrics = summary[dataset]

        # Check if metrics have method level (ECC, GCA) or not
        if any(key in metrics for key in ["ECC", "GCA"]):
            # Has method level (like EMOS, DRN)
            for method, method_metrics in metrics.items():
                if not isinstance(method_metrics, dict):
                    continue  # Skip non-dict values
                for metric_name, time_values in method_metrics.items():
                    if not isinstance(time_values, dict):
                        continue  # Skip non-dict values
                    for time_horizon, value in time_values.items():
                        # Convert time_horizon string (e.g., "24h") to hours integer
                        hours = int(time_horizon.rstrip("h"))
                        rows.append(
                            {
                                "run_id": run_id,
                                "dataset": dataset,
                                "model_name": model_name,
                                "method": method,
                                "pred_method": None,
                                "metric_name": metric_name,
                                "timedelta_hours": hours,
                                "value": value,
                            }
                        )
        else:  # No method level, metrics are directly under dataset
            # Determine loss function for method column, if available
            if json_config.get("model", False) and json_config["model"]["value"].get(
                "loss_fn", False
            ):
                loss_fn = json_config["model"]["value"]["loss_fn"]["_target_"].split(".")[-1]
            else:
                loss_fn = None

            # Determine if direct or indirect predictions
            tags = run.tags
            if "direct" in tags:
                pred_method = "direct"
            elif "indirect" in tags:
                pred_method = "indirect"
            else:
                pred_method = None

            for metric_name, time_values in metrics.items():
                if not isinstance(time_values, dict):
                    continue  # Skip non-dict values (like _wandb metadata)
                for time_horizon, value in time_values.items():
                    # Convert time_horizon string (e.g., "24h") to hours integer
                    hours = int(time_horizon.rstrip("h"))
                    rows.append(
                        {
                            "run_id": run_id,
                            "dataset": dataset,
                            "model_name": model_name,
                            "method": loss_fn,
                            "pred_method": pred_method,
                            "metric_name": metric_name,
                            "timedelta_hours": hours,
                            "value": value,
                        }
                    )

# Create Polars DataFrame with explicit Duration type
df = (
    pl.DataFrame(rows, infer_schema_length=1000)
    .with_columns(
        pl.col("timedelta_hours").mul(3600_000_000).cast(pl.Duration("us")).alias("timedelta"),
        # Rename model_name from CHEN to LNGM for better readability
        pl.col("model_name").replace({"CHEN": "LNGM", "Raw Ensemble": "RAW"}).alias("model_name"),
        # Rename loss fn to their short names for better readability
        pl.col("method")
        .replace(
            {
                "EnergyScore": "ES",
                "PatchwiseEnergyScore": "PES",
                "MultiScaleEnergyScore": "MSES",
                "MultiScalePatchwiseEnergyScore": "MSPES",
            }
        )
        .alias("method"),
    )
    .drop("timedelta_hours")
)

if DATASET == "icon":
    # Rename the metrics for the icon dataset as they slightly differ from the wb2 ones
    df = df.with_columns(
        pl.col("metric_name")
        .str.replace("T_2M+height_2.0", "2m_temperature", literal=True)
        .str.replace("T_2M", "2m_temperature", literal=True)
        .str.replace("2m_temperature+height_2.0", "2m_temperature", literal=True)
        .str.replace("VMAX_10M", "10m_wind_speed", literal=True)
        .str.replace("VMAX_10M+height_2_10.0", "10m_wind_speed", literal=True)
        .str.replace("10m_wind_speed+height_2_10.0", "10m_wind_speed", literal=True)
        .str.replace("crps", "CRPS", literal=True)
        .alias("metric_name"),
    )

df

run_id,dataset,model_name,method,pred_method,metric_name,value,timedelta
str,str,str,str,str,str,f64,duration[μs]
"""iy9kmiv2""","""test""","""FM_UVIT""",null,"""direct""","""CRPS_2m_temperature""",0.855567,4d 12h
"""iy9kmiv2""","""test""","""FM_UVIT""",null,"""direct""","""CRPS_2m_temperature""",0.862443,5d
"""iy9kmiv2""","""test""","""FM_UVIT""",null,"""direct""","""CRPS_2m_temperature""",0.481654,12h
"""iy9kmiv2""","""test""","""FM_UVIT""",null,"""direct""","""CRPS_2m_temperature""",0.491445,1d
"""iy9kmiv2""","""test""","""FM_UVIT""",null,"""direct""","""CRPS_2m_temperature""",0.539995,1d 12h
…,…,…,…,…,…,…,…
"""03xpce4v""","""test""","""ENGRESSION""","""ES""","""indirect""","""PatchwiseEnergyScore_combined""",0.941478,2d
"""03xpce4v""","""test""","""ENGRESSION""","""ES""","""indirect""","""PatchwiseEnergyScore_combined""",1.086528,2d 12h
"""03xpce4v""","""test""","""ENGRESSION""","""ES""","""indirect""","""PatchwiseEnergyScore_combined""",1.139327,3d


In [11]:
df_plot = df.with_columns(
    [
        # Create combined model+method label
        pl.when(pl.col("method").is_not_null() & pl.col("pred_method").is_null())
        .then(pl.col("model_name") + " (" + pl.col("method") + ")")
        .when(pl.col("method").is_not_null() & pl.col("pred_method").is_not_null())
        .then(pl.col("model_name") + " (" + pl.col("method") + ", " + pl.col("pred_method") + ")")
        .when(pl.col("method").is_null() & pl.col("pred_method").is_not_null())
        .then(pl.col("model_name") + " (" + pl.col("pred_method") + ")")
        .otherwise(pl.col("model_name"))
        .alias("model_method"),
        # Keep line style semantic directly in the dataframe
        pl.when(pl.col("pred_method") == "direct")
        .then(pl.lit("direct"))
        .otherwise(pl.lit("indirect"))
        .alias("line_style"),
        # Convert timedelta to days for better readability
        (pl.col("timedelta").dt.total_days(fractional=True)).alias("days"),
        # Split metric_name into score and variable
        pl.col("metric_name")
        .str.split("_")
        .list.first()
        .replace({"EnergyScore": "Energy Score", "VariogramScore": "Variogram Score"})
        .alias("score"),
        pl.col("metric_name").str.split("_").list.slice(1).list.join("_").alias("variable"),
    ]
).sort(["score", "variable", "model_method", "days"])


df_plot = df_plot.filter(pl.col("dataset").eq("test") & pl.col("variable").eq("combined"))
df_plot

run_id,dataset,model_name,method,pred_method,metric_name,value,timedelta,model_method,line_style,days,score,variable
str,str,str,str,str,str,f64,duration[μs],str,str,f64,str,str
"""qmge5ywq""","""test""","""DRN""","""ECC""",null,"""CRPS_combined""",1.12968,12h,"""DRN (ECC)""","""indirect""",0.5,"""CRPS""","""combined"""
"""qmge5ywq""","""test""","""DRN""","""ECC""",null,"""CRPS_combined""",1.135971,1d,"""DRN (ECC)""","""indirect""",1.0,"""CRPS""","""combined"""
"""qmge5ywq""","""test""","""DRN""","""ECC""",null,"""CRPS_combined""",1.184191,1d 12h,"""DRN (ECC)""","""indirect""",1.5,"""CRPS""","""combined"""
"""qmge5ywq""","""test""","""DRN""","""ECC""",null,"""CRPS_combined""",1.185966,2d,"""DRN (ECC)""","""indirect""",2.0,"""CRPS""","""combined"""
"""qmge5ywq""","""test""","""DRN""","""ECC""",null,"""CRPS_combined""",1.255826,2d 12h,"""DRN (ECC)""","""indirect""",2.5,"""CRPS""","""combined"""
…,…,…,…,…,…,…,…,…,…,…,…,…
"""xa1kwv7b""","""test""","""LNGM""","""PES""","""indirect""","""PatchwiseEnergyScore_combined""",1.052706,3d,"""LNGM (PES, indirect)""","""indirect""",3.0,"""PatchwiseEnergyScore""","""combined"""
"""xa1kwv7b""","""test""","""LNGM""","""PES""","""indirect""","""PatchwiseEnergyScore_combined""",1.215392,3d 12h,"""LNGM (PES, indirect)""","""indirect""",3.5,"""PatchwiseEnergyScore""","""combined"""
"""xa1kwv7b""","""test""","""LNGM""","""PES""","""indirect""","""PatchwiseEnergyScore_combined""",1.246927,4d,"""LNGM (PES, indirect)""","""indirect""",4.0,"""PatchwiseEnergyScore""","""combined"""


In [ ]:
to_table = df_plot.group_by("run_id", "score", "model_name", "method", "pred_method").agg(
    pl.col("value").mean()
)

In [19]:
print(to_table)

shape: (59, 6)
┌──────────┬────────────────────────────────┬────────────┬────────┬─────────────┬────────────┐
│ run_id   ┆ score                          ┆ model_name ┆ method ┆ pred_method ┆ value      │
│ ---      ┆ ---                            ┆ ---        ┆ ---    ┆ ---         ┆ ---        │
│ str      ┆ str                            ┆ str        ┆ str    ┆ str         ┆ f64        │
╞══════════╪════════════════════════════════╪════════════╪════════╪═════════════╪════════════╡
│ 24yqcvzc ┆ PatchwiseEnergyScore           ┆ FM_UVIT    ┆ null   ┆ indirect    ┆ 1.053411   │
│ iy9kmiv2 ┆ Energy Score                   ┆ FM_UVIT    ┆ null   ┆ direct      ┆ 461.088324 │
│ xa1kwv7b ┆ PatchwiseEnergyScore           ┆ LNGM       ┆ PES    ┆ indirect    ┆ 1.058997   │
│ n5klic9q ┆ PatchwiseEnergyScore           ┆ FM_UNET    ┆ null   ┆ indirect    ┆ 1.007979   │
│ 3xyv0tvc ┆ Energy Score                   ┆ ENGRESSION ┆ PES    ┆ indirect    ┆ 444.19426  │
│ …        ┆ …                     

In [ ]:
# Build a LaTeX table from to_table with robust metric name normalization
score_name_map = {
    "CRPS": "CRPS",
    "EnergyScore": "EnergyScore",
    "Energy Score": "EnergyScore",
    "PatchwiseEnergyScore": "PatchwiseEnergyScore",
    "Patchwise Energy Score": "PatchwiseEnergyScore",
    "MultiScaleEnergyScore": "MultiScaleEnergyScore",
    "Multi Scale Energy Score": "MultiScaleEnergyScore",
    "MultiScalePatchwiseEnergyScore": "MultiScalePatchwiseEnergyScore",
    "Multi Scale Patchwise Energy Score": "MultiScalePatchwiseEnergyScore",
    "VariogramScore": "VariogramScore",
    "Variogram Score": "VariogramScore",
    "VS": "VariogramScore",
}

score_df = to_table.with_columns(
    pl.col("score").replace(score_name_map).alias("Metric"),
    pl.col("model_name").str.replace("_", "\\_", literal=True).alias("Model"),
    pl.col("method").fill_null("").alias("Loss Fn"),
    pl.col("pred_method").fill_null("").alias("Directness"),
).select(["Model", "Loss Fn", "Directness", "Metric", "value"])

# Aggregate before pivot in case multiple runs map to the same Model/LossFn/Directness
score_df = (
    score_df.group_by(["Model", "Loss Fn", "Directness", "Metric"])
    .agg(pl.col("value").mean().alias("Score"))
    .pivot(on="Metric", index=["Model", "Loss Fn", "Directness"], values="Score")
)

ordered_cols = [
    "CRPS",
    "EnergyScore",
    "PatchwiseEnergyScore",
    "MultiScaleEnergyScore",
    "MultiScalePatchwiseEnergyScore",
    "VariogramScore",
]
score_cols = [col for col in ordered_cols if col in score_df.columns]

best = {}
second_best = {}
for col in score_cols:
    sorted_vals = score_df[col].drop_nulls().sort().to_list()
    if len(sorted_vals) > 0:
        best[col] = sorted_vals[0]
        second_best[col] = sorted_vals[1] if len(sorted_vals) > 1 else None

latex_lines = []
latex_lines.append(r"\begin{table}[ht]")
latex_lines.append(r"\centering")
latex_lines.append(r"\begin{tabular}{lll" + "r" * len(score_cols) + "}")
latex_lines.append(r"\toprule")

col_labels = {
    "CRPS": r"\textbf{CRPS} ($\times 10^{-1}$)",
    "EnergyScore": r"\textbf{ES} ($\times 10^{2}$)",
    "PatchwiseEnergyScore": r"\textbf{PES} ($\times 10^{-1}$)",
    "MultiScaleEnergyScore": r"\textbf{MSES} ($\times 10^{-1}$)",
    "MultiScalePatchwiseEnergyScore": r"\textbf{MSPES} ($\times 10^{-1}$)",
    "VariogramScore": r"\textbf{VS} ($\times 10^{5}$)",
}

scale_factor = {
    "CRPS": 1e-1,
    "EnergyScore": 1e2,
    "PatchwiseEnergyScore": 1e-1,
    "MultiScaleEnergyScore": 1e-1,
    "MultiScalePatchwiseEnergyScore": 1e-1,
    "VariogramScore": 1e5,
}

header = [r"\textbf{Model}", r"\textbf{Loss Fn}", r"\textbf{Direct?}"] + [
    col_labels.get(col, f"\textbf{{{col}}}") for col in score_cols
]
latex_lines.append(" & ".join(header) + r" \\")
latex_lines.append(r"\midrule")

for row in score_df.iter_rows(named=True):
    direct = row.get("Directness", "")
    if direct == "direct":
        direct_mark = r"\ding{51}"
    elif direct == "indirect":
        direct_mark = r"\ding{55}"
    else:
        direct_mark = "--"

    cells = [row["Model"], row.get("Loss Fn", ""), direct_mark]
    for col in score_cols:
        v = row[col]
        if v is None:
            formatted = "--"
        else:
            factor = scale_factor.get(col, 1.0)
            formatted_val = v / factor
            formatted = f"{formatted_val:.2f}"
            if v == best.get(col):
                formatted = r"\textbf{" + formatted + r"}"
            elif v == second_best.get(col):
                formatted = r"\underline{" + formatted + r"}"
        cells.append(formatted)
    latex_lines.append(" & ".join(cells) + r" \\")

latex_lines.append(r"\bottomrule")
latex_lines.append(r"\end{tabular}")
latex_lines.append(r"\caption{Scores table for ICON combined variable}")
latex_lines.append(r"\label{tab:scores_table_icon}")
latex_lines.append(r"\end{table}")

latex_str = "\n".join(latex_lines)
print(latex_str)

\begin{table}[ht]
\centering
\begin{tabular}{lllrrrrr}
\toprule
\textbf{Model} & \textbf{Loss Fn} & \textbf{Direct?} & \textbf{CRPS} ($\times 10^{-1}$) & \textbf{ES} ($\times 10^{1}$) & \textbf{PES} ($\times 10^{-1}$) & \textbf{MSES} ($\times 10^{-1}$) & \textbf{MSPES} ($\times 10^{-1}$) \\
\midrule
FM\_UVIT &  & \ding{55} & 8.89 & 4.52 & 10.53 & 11.90 & 9.82 \\
ENGRESSION & ES & \ding{55} & 10.49 & 4.51 & 11.40 & 12.97 & 10.91 \\
LNGM & PES & \ding{55} & 8.93 & 4.51 & 10.59 & 11.69 & 9.70 \\
EMOS & ECC & -- & 8.70 & 4.45 & 10.43 & \textbf{11.39} & \textbf{9.41} \\
LNGM & ES & \ding{55} & 9.42 & 4.52 & 11.01 & 12.19 & 10.35 \\
ENGRESSION & MSES & \ding{55} & 10.99 & 4.60 & 11.60 & 11.71 & 10.58 \\
LNGM & MSES & \ding{55} & 9.60 & 4.59 & 11.21 & 11.84 & 10.18 \\
FM\_UNET &  & \ding{55} & \textbf{8.51} & \textbf{4.35} & \textbf{10.08} & \underline{11.45} & \underline{9.41} \\
ENGRESSION & PES & \ding{55} & 8.75 & 4.44 & 10.37 & 11.48 & 9.47 \\
FM\_UVIT &  & \ding{51} & 9.09 & 4.61 & -- &